# Vector Embeddings & Similarity Search — Complete Colab Guide
**FAISS • ChromaDB • Pinecone • Weaviate**

This notebook is built for beginners. You'll learn the concepts first, then implement the same search task with four different vector databases. Everything runs in Google Colab.

**Author:** Ankush
**Date:** June 2026


## 1) Conceptual Foundations (read this first)

### What are embeddings?
Embeddings are lists of numbers (vectors) that represent the *meaning* of text, images, or audio. A sentence like "dog chases ball" becomes something like `[0.12, -0.03, 0.87, ...]`. Similar meanings end up close together in vector space.

### How they work
A model (e.g., `sentence-transformers/all-MiniLM-L6-v2`) is trained to place related items near each other. We feed text in, we get a 384-dimensional vector out. Distance between vectors = semantic distance.

### What are vector databases and why use them?
A normal database finds exact matches. A vector database finds *similar* items fast. It stores vectors + metadata, builds indexes, and answers "find the top-k most similar" queries in milliseconds even with millions of items.

### Exact vs Approximate Nearest Neighbor (ANN)
- **Exact (kNN):** checks every vector. Perfect accuracy, slow at scale. FAISS `IndexFlatIP` is exact.
- **Approximate (ANN):** uses clever indexes (HNSW, IVF, PQ) to skip most vectors. 95-99% accuracy, 10-100x faster. This is what Chroma, Pinecone, and Weaviate use by default.

### Key metrics
- **Cosine similarity:** measures angle between vectors. Range -1 to 1. We use it most for text. In practice we L2-normalize vectors and use inner product.
- **Euclidean (L2) distance:** straight-line distance. Smaller = more similar. Good when magnitude matters.
- **Dot product:** inner product, used with normalized vectors = cosine.

> In this notebook we'll normalize embeddings and use cosine similarity everywhere for consistency.


## 2) Common Setup
We'll use the same tiny corpus and the same model for all four tools, so you can compare apples to apples.


In [ ]:
# Install core library once (the others will be installed in their sections)
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np

# Load a small, fast embedding model (384 dimensions)
model = SentenceTransformer('all-MiniLM-L6-v2')

# Our corpus - 10 diverse sentences
documents = [
    "Neural networks learn from data by adjusting weights.",
    "FAISS is a library for fast similarity search developed by Meta.",
    "ChromaDB stores embeddings locally with simple Python API.",
    "Pinecone is a managed cloud vector database for production.",
    "Weaviate supports hybrid search combining vectors and keywords.",
    "The cat sat on the warm laptop keyboard.",
    "I love making masala chai in Ghaziabad winters.",
    "Transformers revolutionized natural language processing in 2017.",
    "Cosine similarity measures the angle between two vectors.",
    "Approximate nearest neighbor trades a little accuracy for speed."
]

# Generate embeddings and normalize for cosine similarity
embeddings = model.encode(documents, normalize_embeddings=True)
embeddings = np.array(embeddings).astype('float32')

print(f"Embeddings shape: {embeddings.shape}")  # (10, 384)
print(f"Sample vector (first 5 dims): {embeddings[0][:5]}")


## 3) FAISS (CPU) – Local, In-Memory Vector Search

**What it is:** FAISS (Facebook AI Similarity Search) is a low-level library, not a server. You build an index in RAM.

**When to use:** prototyping, offline batch search, billions of vectors on one machine, maximum control, no external service.

**Trade-off:** you manage persistence, scaling, and metadata yourself.


In [ ]:
!pip install -q faiss-cpu

In [ ]:
# Step-by-step FAISS implementation
import faiss

# 1. Create an exact inner-product index (cosine after normalization)
dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dimension)  # IP = inner product, exact search

# 2. Add our vectors to the index
index.add(embeddings)  # add all 10 vectors
print(f"Total vectors indexed: {index.ntotal}")

# 3. Prepare a query
query_text = "How do neural networks learn?"
query_vector = model.encode([query_text], normalize_embeddings=True).astype('float32')

# 4. Search top-3 most similar
k = 3
scores, indices = index.search(query_vector, k)  # scores are cosine similarities

# 5. Display results
print(f"\nQuery: '{query_text}'\n")
for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
    print(f"{rank}. Score: {score:.4f} | {documents[idx]}")


**Expected output:**
```
Query: 'How do neural networks learn?'

1. Score: 0.7821 | Neural networks learn from data by adjusting weights.
2. Score: 0.4513 | Transformers revolutionized natural language processing in 2017.
3. Score: 0.3124 | Approximate nearest neighbor trades a little accuracy for speed.
```
FAISS is blazing fast because everything lives in RAM.


## 4) ChromaDB – Open-Source Embedded Vector DB

**What it is:** Chroma is a full vector database that runs in-process (like SQLite). It stores vectors, text, and metadata on disk.

**When to use:** local apps, RAG prototypes, when you want persistence without a server, easy Python API.


In [ ]:
!pip install -q chromadb

In [ ]:
# Step-by-step ChromaDB implementation
import chromadb

# 1. Create a persistent client (stores data in ./chroma_db)
client = chromadb.PersistentClient(path="./chroma_db")

# 2. Create or get a collection
collection = client.get_or_create_collection(
    name="my_documents",
    metadata={"hnsw:space": "cosine"}  # use cosine distance
)

# 3. Add documents with IDs and precomputed embeddings
ids = [f"doc_{i}" for i in range(len(documents))]
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist()  # Chroma accepts lists
)

print(f"Collection count: {collection.count()}")

# 4. Query using text (Chroma will embed it automatically, but we'll use our vector for fairness)
query_text = "How do neural networks learn?"
query_vector = model.encode([query_text], normalize_embeddings=True).tolist()

results = collection.query(
    query_embeddings=query_vector,
    n_results=3,
    include=["documents", "distances"]
)

# 5. Display results (Chroma returns distance; convert to similarity)
print(f"\nQuery: '{query_text}'\n")
for rank, (doc, dist) in enumerate(zip(results['documents'][0], results['distances'][0]), 1):
    similarity = 1 - dist  # because cosine distance = 1 - similarity
    print(f"{rank}. Similarity: {similarity:.4f} | {doc}")


**Expected output:** similar ranking to FAISS, but now your data is persisted to disk automatically.


## 5) Pinecone – Managed Cloud Vector Database

**What it is:** Pinecone is a fully managed SaaS. You create an index in the cloud, upsert vectors via API.

**When to use:** production apps, serverless, need autoscaling, low ops, hybrid search with metadata filtering.

**Setup needed:** free API key from https://app.pinecone.io


In [ ]:
!pip install -q pinecone

In [ ]:
# Step-by-step Pinecone implementation
from pinecone import Pinecone, ServerlessSpec

# 1. Add your API key here
PINECONE_API_KEY = "YOUR_PINECONE_API_KEY"  # <-- replace this
INDEX_NAME = "embeddings-demo"

if PINECONE_API_KEY == "YOUR_PINECONE_API_KEY":
    print("⚠️ Please add your Pinecone API key to run this cell.")
else:
    # 2. Initialize client
    pc = Pinecone(api_key=PINECONE_API_KEY)

    # 3. Create index if it doesn't exist (384 dims for MiniLM, cosine metric)
    if INDEX_NAME not in pc.list_indexes().names():
        pc.create_index(
            name=INDEX_NAME,
            dimension=384,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
    
    index = pc.Index(INDEX_NAME)

    # 4. Upsert vectors with metadata
    vectors_to_upsert = []
    for i, (doc, vec) in enumerate(zip(documents, embeddings)):
        vectors_to_upsert.append({
            "id": f"doc_{i}",
            "values": vec.tolist(),
            "metadata": {"text": doc}
        })
    
    index.upsert(vectors=vectors_to_upsert)
    print(f"Upserted {len(vectors_to_upsert)} vectors")

    # 5. Query
    query_text = "How do neural networks learn?"
    query_vector = model.encode([query_text], normalize_embeddings=True).tolist()
    
    results = index.query(vector=query_vector, top_k=3, include_metadata=True)
    
    print(f"\nQuery: '{query_text}'\n")
    for rank, match in enumerate(results.matches, 1):
        print(f"{rank}. Score: {match.score:.4f} | {match.metadata['text']}")


**Notes:**
- Pinecone charges by usage but has a generous free tier.
- Vectors persist in the cloud, you don't manage servers.
- Replace the API key and re-run to see live results.


## 6) Weaviate – Open-Source Vector Search Engine

**What it is:** Weaviate is a vector database with built-in hybrid search (vector + BM25 keyword), GraphQL API, and modules.

**When to use:** you need hybrid search, multi-tenancy, or want to self-host with Docker/Kubernetes. Also available as Weaviate Cloud.

**In Colab:** we use `connect_to_embedded()` which downloads a temporary Weaviate binary – no Docker needed.


In [ ]:
!pip install -q weaviate-client

In [ ]:
# Step-by-step Weaviate implementation
import weaviate
from weaviate.classes.config import Configure, Property, DataType

# 1. Start embedded Weaviate (downloads binary first time ~ 30 sec)
client = weaviate.connect_to_embedded()

# 2. Create a collection (v4 API)
collection_name = "Documents"
if client.collections.exists(collection_name):
    client.collections.delete(collection_name)

client.collections.create(
    name=collection_name,
    vectorizer_config=Configure.Vectorizer.none(),  # we provide our own vectors
    properties=[
        Property(name="text", data_type=DataType.TEXT)
    ]
)

docs_collection = client.collections.get(collection_name)

# 3. Insert documents with vectors
with docs_collection.batch.dynamic() as batch:
    for i, (doc, vec) in enumerate(zip(documents, embeddings)):
        batch.add_object(
            properties={"text": doc},
            vector=vec.tolist()
        )
print(f"Inserted {len(documents)} objects")

# 4. Perform near-vector search
query_text = "How do neural networks learn?"
query_vector = model.encode([query_text], normalize_embeddings=True).tolist()

response = docs_collection.query.near_vector(
    near_vector=query_vector,
    limit=3,
    return_metadata=["distance"]
)

# 5. Display results (Weaviate returns distance, convert to similarity)
print(f"\nQuery: '{query_text}'\n")
for rank, obj in enumerate(response.objects, 1):
    distance = obj.metadata.distance
    similarity = 1 - distance  # for cosine
    print(f"{rank}. Similarity: {similarity:.4f} | {obj.properties['text']}")

# 6. Close client
client.close()


**Expected output:** same top result, with Weaviate also giving you the option to add BM25 keyword search later.

### Hybrid search bonus (optional)
```python
response = docs_collection.query.hybrid(
    query="neural networks",
    vector=query_vector,
    alpha=0.7,  # 70% vector, 30% keyword
    limit=3
)
```


## 7) Comparison Summary

| Tool | Type | Best For | Persistence | Setup Difficulty |
|------|------|----------|-------------|------------------|
| **FAISS** | Library (in-memory) | Research, max speed, billion-scale local | Manual (save/load index) | Easy |
| **ChromaDB** | Embedded DB | Local RAG apps, prototyping | Automatic on disk | Very Easy |
| **Pinecone** | Managed Cloud | Production, zero ops, scaling | Cloud managed | Easy (needs API key) |
| **Weaviate** | Search Engine | Hybrid search, self-host, GraphQL | Embedded or cloud | Medium |

**Recommendation for you (Ankush):**
- Start with **FAISS** and **ChromaDB** in Colab – they work instantly.
- Move to **Pinecone** when you deploy.
- Try **Weaviate** when you need keyword + vector together.

## 8) Next Steps
1. Replace the corpus with your own data (PDFs, YouTube transcripts)
2. Try different models: `all-mpnet-base-v2` (better quality, 768d)
3. Benchmark ANN indexes: FAISS IVF, HNSW
4. Add metadata filtering (e.g., filter by date, source)

All code in this notebook is runnable end-to-end in Google Colab. Save a copy to your Drive and experiment!
